In [9]:
import requests
import json
import pandas as pd
import copy
import time
import requests
import pandas as pd
import json
import copy
import time

In [38]:
def generar_meses():
    meses = ["ene", "feb", "mar", "abr", "may", "jun",
             "jul", "ago", "sep", "oct", "nov", "dic"]
    
    resultado = []
    for anio in range(2015, 2026):
        for mes in meses:
            resultado.append(f"{mes}. {str(anio)[-2:]}")
    
    return resultado


def parse_looker_response(data, mes):
    dataset = data['dataResponse'][0]['dataSubset'][0]['dataset']['tableDataset']
    
    columns_list = dataset['column']
    total_rows = dataset['size']

    final_columns = {}

    for i, col_data in enumerate(columns_list):

        col_name = col_data.get('name') or f"col_{i}"

        key = 'stringColumn' if 'stringColumn' in col_data else 'doubleColumn'
        values = col_data.get(key, {}).get('values', [])
        null_indices = col_data.get('nullIndex', [])

        full_col = []
        val_ptr = 0

        for idx in range(total_rows):
            if idx in null_indices:
                full_col.append(None)
            else:
                full_col.append(values[val_ptr])
                val_ptr += 1

        final_columns[col_name] = full_col

    df = pd.DataFrame(final_columns)
    df["mes"] = mes

    return df


# 🔥 NUEVA FUNCIÓN
def find_mes_filter(payload):
    filters = payload['dataRequest'][0]['datasetSpec']['filters']
    
    for i, f in enumerate(filters):
        try:
            exp = f['filterDefinition']['filterExpression']
            
            if 'stringValues' in exp:
                val = exp['stringValues'][0]
                
                # detecta formato tipo "ene. 24"
                if isinstance(val, str) and "." in val:
                    return i
        except:
            continue
    
    raise ValueError("No se encontró el filtro de mes")


def run_looker_pipeline(
    cookies, 
    headers, 
    params, 
    json_data,
    pais=None,
    ciudad=None,
    operacion=None,
    columnas=None,
    mes_filter_idx=None,        # 🔥 NUEVO
    mes_field_name=None         # 🔥 OPCIONAL (ej: "_fecha_")
):
    
    meses = generar_meses()
    dfs = []

    # ==============================
    # 🔥 DETECCIÓN DEL FILTRO
    # ==============================
    if mes_filter_idx is None:
        filters = json_data['dataRequest'][0]['datasetSpec']['filters']
        
        for i, f in enumerate(filters):
            try:
                field = f['filterDefinition']['filterExpression']['queryTimeTransformation']['dataTransformation']['sourceFieldName']
                
                if mes_field_name and field == mes_field_name:
                    mes_filter_idx = i
                    break
            except:
                continue

        if mes_filter_idx is None:
            raise ValueError("No se encontró el filtro de mes")

    print(f"📍 Usando filtro de mes en índice: {mes_filter_idx}")

    # ==============================
    # LOOP
    # ==============================
    for mes in meses:
        try:
            payload = copy.deepcopy(json_data)

            payload['dataRequest'][0]['datasetSpec']['filters'][mes_filter_idx]['filterDefinition']['filterExpression']['stringValues'] = [mes]

            response = requests.post(
                'https://lookerstudio.google.com/embed/u/0/batchedDataV2',
                params=params,
                cookies=cookies,
                headers=headers,
                json=payload,
            )

            if response.status_code != 200:
                print(f"❌ Error en {mes}")
                continue

            raw_str = response.text[5:].strip()
            data = json.loads(raw_str)

            df = parse_looker_response(data, mes)

            if len(df) == 0:
                print(f"⚠️ {mes} sin datos")
                continue

            # ==============================
            # METADATOS
            # ==============================
            if pais is not None:
                df["pais"] = pais
            if ciudad is not None:
                df["ciudad"] = ciudad
            if operacion is not None:
                df["operacion"] = operacion

            # ==============================
            # RENOMBRE
            # ==============================
            if columnas is not None:
                df = df.rename(columns=columnas)

            dfs.append(df)

            print(f"✅ {mes} | {len(df)} filas")

            time.sleep(1)

        except Exception as e:
            print(f"⚠️ Error en {mes}: {e}")

    if dfs:
        return pd.concat(dfs, ignore_index=True)
    else:
        return pd.DataFrame()

### CDMX

#### renta

In [16]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AK9w0o9EjP-29wkGm_oluyxY5hKHw:1774506119790',
    '__Secure-BUCKET': 'CJoE',
    'SEARCH_SAMESITE': 'CgQIraAB',
    '_gid': 'GA1.3.880237846.1774295282',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=',
    'SID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zsL8Kmlc9hKwa0QlGnWAM9QACgYKASQSARASFQHGX2MiurgzhV04eZJDvQB7CX5llRoVAUF8yKoe1iQuk7_Bw8p0uyvfpDZK0076',
    '__Secure-1PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zmMqCj_uXYQahrclVpkmjrAACgYKAV8SARASFQHGX2MidUqxNKYSorSOySMKnt-VRBoVAUF8yKpBAyLm2AbpmioFR39ZsI5U0076',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    'HSID': 'AAZCO_3EZJb0oSvR4',
    'SSID': 'AyGp0W0zZhC3StMy4',
    'APISID': 'OMjJovW-w2ce5Dyt/ALv5EMReHF1u1d-rS',
    'SAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '__Secure-1PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '_ga_RWLCE6YRP6': 'GS2.3.s1774446646$o10$g0$t1774446646$j60$l0$h0',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    'AEC': 'AaJma5unWxIWKAETJomwJ-4GtSWvWAB7YtXrThYSdXszSsIw__vPKafX6g',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-1PSIDTS': 'sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA',
    '_gat': '1',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=',
    '_ga': 'GA1.3.966341472.1774295282',
    '_gat_marketingTracker': '1',
    'SIDCC': 'AKEyXzXCJoCswsWP7c3uV01JgoZYWzIqK2AJ5Blis-JYpSVeFZoLMNpVR0gyCVo94ttbu35tvZLB',
    '__Secure-1PSIDCC': 'AKEyXzV1JozhKSY6J2cFo4jtDd0GeL-dCTXs0uQL3CwQuYJwlT7JmgnCvlBo_YP1LfbL5FvKX8yc',
    '__Secure-3PSIDCC': 'AKEyXzVJxGtRPXjNrVY5z6pj9HfGnQDb7IZRaWIJAE65yoBx9Gbz6EEtTu4Uok13Dg1qmMEACZ_MdQ',
    '_ga_S4FJY0X3VX': 'GS2.1.s1774503812$o17$g1$t1774506142$j38$l0$h0',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/2ec3057e-e0dd-42be-9241-a909d1ff3ee6/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BCIK0zwEI1rfPARixis8BGOyxzwE=',
    'x-rap-xsrf-token': 'AImk1AK9w0o9EjP-29wkGm_oluyxY5hKHw:1774506119790',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AK9w0o9EjP-29wkGm_oluyxY5hKHw:1774506119790; __Secure-BUCKET=CJoE; SEARCH_SAMESITE=CgQIraAB; _gid=GA1.3.880237846.1774295282; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=; SID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zsL8Kmlc9hKwa0QlGnWAM9QACgYKASQSARASFQHGX2MiurgzhV04eZJDvQB7CX5llRoVAUF8yKoe1iQuk7_Bw8p0uyvfpDZK0076; __Secure-1PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zmMqCj_uXYQahrclVpkmjrAACgYKAV8SARASFQHGX2MidUqxNKYSorSOySMKnt-VRBoVAUF8yKpBAyLm2AbpmioFR39ZsI5U0076; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; HSID=AAZCO_3EZJb0oSvR4; SSID=AyGp0W0zZhC3StMy4; APISID=OMjJovW-w2ce5Dyt/ALv5EMReHF1u1d-rS; SAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; __Secure-1PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; _ga_RWLCE6YRP6=GS2.3.s1774446646$o10$g0$t1774446646$j60$l0$h0; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AEC=AaJma5unWxIWKAETJomwJ-4GtSWvWAB7YtXrThYSdXszSsIw__vPKafX6g; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-1PSIDTS=sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA; __Secure-3PSIDTS=sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA; _gat=1; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=; _ga=GA1.3.966341472.1774295282; _gat_marketingTracker=1; SIDCC=AKEyXzXCJoCswsWP7c3uV01JgoZYWzIqK2AJ5Blis-JYpSVeFZoLMNpVR0gyCVo94ttbu35tvZLB; __Secure-1PSIDCC=AKEyXzV1JozhKSY6J2cFo4jtDd0GeL-dCTXs0uQL3CwQuYJwlT7JmgnCvlBo_YP1LfbL5FvKX8yc; __Secure-3PSIDCC=AKEyXzVJxGtRPXjNrVY5z6pj9HfGnQDb7IZRaWIJAE65yoBx9Gbz6EEtTu4Uok13Dg1qmMEACZ_MdQ; _ga_S4FJY0X3VX=GS2.1.s1774503812$o17$g1$t1774506142$j38$l0$h0',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '2ec3057e-e0dd-42be-9241-a909d1ff3ee6',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '8195b1fb-609b-4c7d-afc8-88810deb3e48',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_imb9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_oz68k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_1a98k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_u298k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_1a98k9zjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'CDMX',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_lk69k9zjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'sep. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_b1g9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [ ]:
cdmx_renta = run_looker_pipeline(cookies, headers, params, json_data, pais="Mexico",ciudad="Mexico City",operacion="rent", 
                                mes_field_name="_fecha_",
                                 
                                 columnas={"col_0":"neighborhood",
                                                                                                                                     "col_1":"new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"
                                                                                                                                     })
print("Total de registros", len(cdmx_renta))
cdmx_renta.to_csv("cdmx_renta.csv", encoding="latin1",index=False)
cdmx_renta

⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin datos
⚠️ nov. 18 sin datos
✅ dic. 18 | 2

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,FRACC CLUB DE GOLF BOSQUES,NaN,24278.0,24717.0,dic. 18,Mexico,Mexico City,rent
1,AMPL GRANADA,23698.0,23409.0,23355.0,dic. 18,Mexico,Mexico City,rent
2,LOS MORALES SECC ALAMEDA,24714.0,23358.0,22994.0,dic. 18,Mexico,Mexico City,rent
3,BOSQUES DE CHAPULTEPEC,31218.0,23096.0,22194.0,dic. 18,Mexico,Mexico City,rent
4,POLANCO REFORMA,28217.0,22460.0,21052.0,dic. 18,Mexico,Mexico City,rent
...,...,...,...,...,...,...,...,...
14521,HACIENDA DE LAS PALMAS,14218.0,14025.0,14068.0,oct. 25,Mexico,Mexico City,rent
14522,VILLA FLORENCE,NaN,12980.0,13008.0,oct. 25,Mexico,Mexico City,rent
14523,NAVIDAD,NaN,12786.0,NaN,oct. 25,Mexico,Mexico City,rent
14524,PANTITLAN,NaN,11668.0,NaN,oct. 25,Mexico,Mexico City,rent


#### CDMX 
##### Venta

In [13]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AK0-q-PaQYuhKVCHoOQUOjqQ7uAIg:1774505724135',
    '__Secure-BUCKET': 'CJoE',
    'SEARCH_SAMESITE': 'CgQIraAB',
    '_gid': 'GA1.3.880237846.1774295282',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=',
    'SID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zsL8Kmlc9hKwa0QlGnWAM9QACgYKASQSARASFQHGX2MiurgzhV04eZJDvQB7CX5llRoVAUF8yKoe1iQuk7_Bw8p0uyvfpDZK0076',
    '__Secure-1PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zmMqCj_uXYQahrclVpkmjrAACgYKAV8SARASFQHGX2MidUqxNKYSorSOySMKnt-VRBoVAUF8yKpBAyLm2AbpmioFR39ZsI5U0076',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    'HSID': 'AAZCO_3EZJb0oSvR4',
    'SSID': 'AyGp0W0zZhC3StMy4',
    'APISID': 'OMjJovW-w2ce5Dyt/ALv5EMReHF1u1d-rS',
    'SAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '__Secure-1PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '_ga_RWLCE6YRP6': 'GS2.3.s1774446646$o10$g0$t1774446646$j60$l0$h0',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    'AEC': 'AaJma5unWxIWKAETJomwJ-4GtSWvWAB7YtXrThYSdXszSsIw__vPKafX6g',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-1PSIDTS': 'sidts-CjEBWhotCdHBSlsgGQmWEyG50fXC56HtaNtfptiRfTq3VQbatIA1eA3NibrEmBnRvlXhEAA',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCdHBSlsgGQmWEyG50fXC56HtaNtfptiRfTq3VQbatIA1eA3NibrEmBnRvlXhEAA',
    '_gat': '1',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=',
    '_ga': 'GA1.3.966341472.1774295282',
    '_gat_marketingTracker': '1',
    'SIDCC': 'AKEyXzVwQSLoGBC4U-SkMEXiQQW5EqLWYvqNFY4JmbJG2-IL7rTNKNxIC7uMlT7f1vK-y0OtzEIk',
    '__Secure-1PSIDCC': 'AKEyXzUqKMKv14PopoFSBW9ElURskE20hCzg-84DBYOdFJAF0xyfOm9tgVn0ojwABO2hhTPgODqz',
    '__Secure-3PSIDCC': 'AKEyXzXIIcPhi2v7B7wMFttD1OBD0I2pj_i5LKgd7Hx3TOSBO5foAFmHrUQknP8coKBGRhQ6iuHLTw',
    '_ga_S4FJY0X3VX': 'GS2.1.s1774503812$o17$g1$t1774505748$j36$l0$h0',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/9d2dfa6f-fae0-4ef6-bb34-e0f1f9415451/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BCIK0zwEI1rfPARixis8BGOyxzwE=',
    'x-rap-xsrf-token': 'AImk1AK0-q-PaQYuhKVCHoOQUOjqQ7uAIg:1774505724135',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AK0-q-PaQYuhKVCHoOQUOjqQ7uAIg:1774505724135; __Secure-BUCKET=CJoE; SEARCH_SAMESITE=CgQIraAB; _gid=GA1.3.880237846.1774295282; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=; SID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zsL8Kmlc9hKwa0QlGnWAM9QACgYKASQSARASFQHGX2MiurgzhV04eZJDvQB7CX5llRoVAUF8yKoe1iQuk7_Bw8p0uyvfpDZK0076; __Secure-1PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zmMqCj_uXYQahrclVpkmjrAACgYKAV8SARASFQHGX2MidUqxNKYSorSOySMKnt-VRBoVAUF8yKpBAyLm2AbpmioFR39ZsI5U0076; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; HSID=AAZCO_3EZJb0oSvR4; SSID=AyGp0W0zZhC3StMy4; APISID=OMjJovW-w2ce5Dyt/ALv5EMReHF1u1d-rS; SAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; __Secure-1PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; _ga_RWLCE6YRP6=GS2.3.s1774446646$o10$g0$t1774446646$j60$l0$h0; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AEC=AaJma5unWxIWKAETJomwJ-4GtSWvWAB7YtXrThYSdXszSsIw__vPKafX6g; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-1PSIDTS=sidts-CjEBWhotCdHBSlsgGQmWEyG50fXC56HtaNtfptiRfTq3VQbatIA1eA3NibrEmBnRvlXhEAA; __Secure-3PSIDTS=sidts-CjEBWhotCdHBSlsgGQmWEyG50fXC56HtaNtfptiRfTq3VQbatIA1eA3NibrEmBnRvlXhEAA; _gat=1; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=; _ga=GA1.3.966341472.1774295282; _gat_marketingTracker=1; SIDCC=AKEyXzVwQSLoGBC4U-SkMEXiQQW5EqLWYvqNFY4JmbJG2-IL7rTNKNxIC7uMlT7f1vK-y0OtzEIk; __Secure-1PSIDCC=AKEyXzUqKMKv14PopoFSBW9ElURskE20hCzg-84DBYOdFJAF0xyfOm9tgVn0ojwABO2hhTPgODqz; __Secure-3PSIDCC=AKEyXzXIIcPhi2v7B7wMFttD1OBD0I2pj_i5LKgd7Hx3TOSBO5foAFmHrUQknP8coKBGRhQ6iuHLTw; _ga_S4FJY0X3VX=GS2.1.s1774503812$o17$g1$t1774505748$j36$l0$h0',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '9d2dfa6f-fae0-4ef6-bb34-e0f1f9415451',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '4f53dde5-ccd3-47a8-a13a-1683958b3094',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_u5wcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_vfpcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_duucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_euucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_duucznzjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'CDMX',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_mspdznzjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'sep. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_vs1cznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [ ]:
cdmx_venta = run_looker_pipeline(cookies, headers, params, json_data, pais="Mexico",ciudad="Mexico City",operacion="sell",  
                                mes_field_name="_fecha_",
                                 columnas={
                                                                                                                                     "col_0":"neighborhood",
                                                                                                                                     "col_1": "new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"  
                                                                                                                                     })
print("Total de registros", len(cdmx_venta))
cdmx_venta.to_csv("cdmx_venta.csv", encoding="latin1",index=False)
cdmx_venta

⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin datos
⚠️ nov. 18 sin datos
✅ dic. 18 | 6

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,HIPODROMO DE LA CONDESA,81726.0,77463.0,73371.0,dic. 18,Mexico,Mexico City,sell
1,ARTURO GAMIZ,NaN,75346.0,72768.0,dic. 18,Mexico,Mexico City,sell
2,CHAPULTEPEC MORALES,89746.0,73869.0,67853.0,dic. 18,Mexico,Mexico City,sell
3,POLANCO REFORMA,78295.0,72182.0,65515.0,dic. 18,Mexico,Mexico City,sell
4,2DA SECC DEL BOSQUE DE CHAPULTEPEC,75074.0,71701.0,71467.0,dic. 18,Mexico,Mexico City,sell
...,...,...,...,...,...,...,...,...
37668,SANTIAGO AHUIZOTLA,NaN,15356.0,15356.0,oct. 25,Mexico,Mexico City,sell
37669,CANUTILLO,NaN,15230.0,15295.0,oct. 25,Mexico,Mexico City,sell
37670,SANTA ANITA,NaN,14565.0,13853.0,oct. 25,Mexico,Mexico City,sell
37671,JACARANDAS,NaN,11781.0,13174.0,oct. 25,Mexico,Mexico City,sell


#### Monterrey 
#### Venta

In [18]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1ALnOEvOJ_xBmzk5rSTYnf9MZy0npw:1774506416686',
    '__Secure-BUCKET': 'CJoE',
    'SEARCH_SAMESITE': 'CgQIraAB',
    '_gid': 'GA1.3.880237846.1774295282',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=',
    'SID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zsL8Kmlc9hKwa0QlGnWAM9QACgYKASQSARASFQHGX2MiurgzhV04eZJDvQB7CX5llRoVAUF8yKoe1iQuk7_Bw8p0uyvfpDZK0076',
    '__Secure-1PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zmMqCj_uXYQahrclVpkmjrAACgYKAV8SARASFQHGX2MidUqxNKYSorSOySMKnt-VRBoVAUF8yKpBAyLm2AbpmioFR39ZsI5U0076',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    'HSID': 'AAZCO_3EZJb0oSvR4',
    'SSID': 'AyGp0W0zZhC3StMy4',
    'APISID': 'OMjJovW-w2ce5Dyt/ALv5EMReHF1u1d-rS',
    'SAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '__Secure-1PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    '_ga_RWLCE6YRP6': 'GS2.3.s1774446646$o10$g0$t1774446646$j60$l0$h0',
    'AMP_MKTG_8f1ede8e9c': 'JTdCJTdE',
    'AEC': 'AaJma5unWxIWKAETJomwJ-4GtSWvWAB7YtXrThYSdXszSsIw__vPKafX6g',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-1PSIDTS': 'sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA',
    '_gat': '1',
    'AMP_8f1ede8e9c': 'JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=',
    '_ga': 'GA1.3.966341472.1774295282',
    '_gat_marketingTracker': '1',
    'SIDCC': 'AKEyXzWnhG0TCETaiQPRSK81XR2l5hXqdT9zQeyLnFuToEbrB9ivmbph8-sbswjot2I0Lu9Vld4c',
    '__Secure-1PSIDCC': 'AKEyXzVrd6nsIsPkBmDL01vJBx86nW9NhUWs13HVnTWm8YC_9_anLrab5KxfVeB1-95p87e-mch0',
    '__Secure-3PSIDCC': 'AKEyXzVa4-t-iUrD2UzbQkTFFsgKI6Ywh-esP781r3A81krtS8vDsSW0UnINzumzcd19gG_5_vJDEQ',
    '_ga_S4FJY0X3VX': 'GS2.1.s1774503812$o17$g1$t1774506445$j32$l0$h0',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/4121d52c-e9af-4584-8094-6bd77864a8b8/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BCIK0zwEI1rfPARixis8BGOyxzwE=',
    'x-rap-xsrf-token': 'AImk1ALnOEvOJ_xBmzk5rSTYnf9MZy0npw:1774506416686',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1ALnOEvOJ_xBmzk5rSTYnf9MZy0npw:1774506416686; __Secure-BUCKET=CJoE; SEARCH_SAMESITE=CgQIraAB; _gid=GA1.3.880237846.1774295282; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=; SID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zsL8Kmlc9hKwa0QlGnWAM9QACgYKASQSARASFQHGX2MiurgzhV04eZJDvQB7CX5llRoVAUF8yKoe1iQuk7_Bw8p0uyvfpDZK0076; __Secure-1PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zmMqCj_uXYQahrclVpkmjrAACgYKAV8SARASFQHGX2MidUqxNKYSorSOySMKnt-VRBoVAUF8yKpBAyLm2AbpmioFR39ZsI5U0076; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; HSID=AAZCO_3EZJb0oSvR4; SSID=AyGp0W0zZhC3StMy4; APISID=OMjJovW-w2ce5Dyt/ALv5EMReHF1u1d-rS; SAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; __Secure-1PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; _ga_RWLCE6YRP6=GS2.3.s1774446646$o10$g0$t1774446646$j60$l0$h0; AMP_MKTG_8f1ede8e9c=JTdCJTdE; AEC=AaJma5unWxIWKAETJomwJ-4GtSWvWAB7YtXrThYSdXszSsIw__vPKafX6g; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-1PSIDTS=sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA; __Secure-3PSIDTS=sidts-CjEBWhotCRjnE_RPxSv070md0qZHBF38WzUMtqBEmRucROQ1hw2VVDPlFw_BOHTAPZC-EAA; _gat=1; AMP_8f1ede8e9c=JTdCJTIyZGV2aWNlSWQlMjIlM0ElMjI2NmYyZTYxZS1hMTFjLTRkZjMtODA3Yy01ZTEzNDY1MDgwMDUlMjIlMkMlMjJ1c2VySWQlMjIlM0ElMjI5ODg3NmJlZS00NmZhLTRiNjUtYjA1My1kMTZmMjVlNTUyMzQlMjIlMkMlMjJzZXNzaW9uSWQlMjIlM0ExNzc0Mjk3NTQ4ODUxJTJDJTIyb3B0T3V0JTIyJTNBZmFsc2UlN0Q=; _ga=GA1.3.966341472.1774295282; _gat_marketingTracker=1; SIDCC=AKEyXzWnhG0TCETaiQPRSK81XR2l5hXqdT9zQeyLnFuToEbrB9ivmbph8-sbswjot2I0Lu9Vld4c; __Secure-1PSIDCC=AKEyXzVrd6nsIsPkBmDL01vJBx86nW9NhUWs13HVnTWm8YC_9_anLrab5KxfVeB1-95p87e-mch0; __Secure-3PSIDCC=AKEyXzVa4-t-iUrD2UzbQkTFFsgKI6Ywh-esP781r3A81krtS8vDsSW0UnINzumzcd19gG_5_vJDEQ; _ga_S4FJY0X3VX=GS2.1.s1774503812$o17$g1$t1774506445$j32$l0$h0',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '4121d52c-e9af-4584-8094-6bd77864a8b8',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '4f53dde5-ccd3-47a8-a13a-1683958b3094',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_u5wcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_vfpcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_duucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_euucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_duucznzjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'MTREY',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_mspdznzjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'may. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_vs1cznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [ ]:
mty_venta = run_looker_pipeline(cookies, headers, params, json_data, pais="Mexico",ciudad="Monterrey",operacion="sell",
                                
                                mes_field_name="_fecha_",
                                 columnas={
                                                                                                                                     "col_0":"neighborhood",
                                                                                                                                     "col_1": "new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"  
                                                                                                                                     })
print("Total de registros", len(mty_venta))
mty_venta.to_csv("mty_venta.csv", encoding="latin1",index=False)
mty_venta

⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin datos
⚠️ nov. 18 sin datos
⚠️ dic. 18 si

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,VALLE DE CAMPESTRE,89478.0,88078.0,77721.0,jul. 21,Mexico,Monterrey,sell
1,VALLE ORIENTE,89604.0,86775.0,78743.0,jul. 21,Mexico,Monterrey,sell
2,PARQUE CORPORATIVO SANTA ENGRACIA,82119.0,83036.0,68763.0,jul. 21,Mexico,Monterrey,sell
3,PUNTO CENTRAL,81663.0,81530.0,NaN,jul. 21,Mexico,Monterrey,sell
4,SAN AGUSTIN CAMPESTRE,79689.0,79450.0,NaN,jul. 21,Mexico,Monterrey,sell
...,...,...,...,...,...,...,...,...
1978,VALLE INFONAVIT SECT 3,NaN,45014.0,NaN,jun. 25,Mexico,Monterrey,sell
1979,DEL VALLE SECTOR ORIENTE,NaN,42598.0,NaN,jun. 25,Mexico,Monterrey,sell
1980,VICTORIA,41242.0,41502.0,NaN,jun. 25,Mexico,Monterrey,sell
1981,RINCON DE LAS HUERTAS,NaN,39754.0,NaN,jun. 25,Mexico,Monterrey,sell


#### Monterrey 
#### Renta

In [20]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AII_15JmTxSTGOxSfc1W_X8tjh80Q:1774506724181',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-3PSIDTS': 'sidts-CjEBBj1CYlQbL6j281YnrRwSBeNjlcvT7Ih3q1L6z-HILKkJMr--OAMJCNVui2WUIw7VEAA',
    '__Secure-3PSIDCC': 'AKEyXzX7WkDCtdzQvdHkbcm3us0USu9_G8jGFA6pv7gtqffP5rHjcgjttDznKriOxDWslMv8A27CXw',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/99c5359b-f9c5-44ca-b766-73f34666d2f9/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AII_15JmTxSTGOxSfc1W_X8tjh80Q:1774506724181',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AII_15JmTxSTGOxSfc1W_X8tjh80Q:1774506724181; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-3PSIDTS=sidts-CjEBBj1CYlQbL6j281YnrRwSBeNjlcvT7Ih3q1L6z-HILKkJMr--OAMJCNVui2WUIw7VEAA; __Secure-3PSIDCC=AKEyXzX7WkDCtdzQvdHkbcm3us0USu9_G8jGFA6pv7gtqffP5rHjcgjttDznKriOxDWslMv8A27CXw',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': '99c5359b-f9c5-44ca-b766-73f34666d2f9',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '8195b1fb-609b-4c7d-afc8-88810deb3e48',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_imb9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_oz68k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_1a98k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_u298k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_1a98k9zjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'MTREY',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_lk69k9zjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'may. 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_b1g9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [ ]:
mty_renta = run_looker_pipeline(cookies, headers, params, json_data, pais="Mexico",ciudad="Monterrey",operacion="rent",
     mes_field_name="_fecha_",
     columnas={
                                                                                                                                     "col_0":"neighborhood",
                                                                                                                                     "col_1": "new",
                                                                                                                                     "col_2": "index",
                                                                                                                                     "col_3": "used"  
                                                                                                                                     })
print("Total de registros", len(mty_renta))
mty_renta.to_csv("mty_renta.csv", encoding="latin1",index=False)
mty_renta

⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin datos
⚠️ nov. 18 sin datos
⚠️ dic. 18 si

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,VALLE ORIENTE,21814.0,20967,20215.0,jul. 21,Mexico,Monterrey,rent
1,VILLAS DE SAN AGUSTIN,20248.0,20312,20341.0,jul. 21,Mexico,Monterrey,rent
2,JARDINES DEL CAMPESTRE,NaN,19570,18630.0,jul. 21,Mexico,Monterrey,rent
3,ALTUS,NaN,18989,NaN,jul. 21,Mexico,Monterrey,rent
4,VALLE DE CAMPESTRE,19436.0,18889,NaN,jul. 21,Mexico,Monterrey,rent
...,...,...,...,...,...,...,...,...
947,MAS PALOMAS,NaN,17972,17972.0,jun. 25,Mexico,Monterrey,rent
948,RES DINASTIA,NaN,17829,18065.0,jun. 25,Mexico,Monterrey,rent
949,VISTA HERMOSA,17860.0,17253,NaN,jun. 25,Mexico,Monterrey,rent
950,MIRADOR RESIDENCIAL,NaN,15616,NaN,jun. 25,Mexico,Monterrey,rent


### Guadalajara
#### Venta

In [39]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1ALJefNbaKsF4_lSHtZw_Lq0HpCgYA:1774508628540',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-3PSIDTS': 'sidts-CjEBBj1CYpmn6P1s58aATI-xiIajUkkEbErtAecu0oUoEyLv8MFPdk1AYzKFT9XDK3eaEAA',
    '__Secure-3PSIDCC': 'AKEyXzVORUAsxRgQaswJv9U474TK67FTluaTIT4T9FbZwMy_rVyl4jrSpiO_zYnR87k6xR96MoNmKA',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/ab865df8-5652-4545-8d4b-0d8c16d71bcf/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1ALJefNbaKsF4_lSHtZw_Lq0HpCgYA:1774508628540',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1ALJefNbaKsF4_lSHtZw_Lq0HpCgYA:1774508628540; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-3PSIDTS=sidts-CjEBBj1CYpmn6P1s58aATI-xiIajUkkEbErtAecu0oUoEyLv8MFPdk1AYzKFT9XDK3eaEAA; __Secure-3PSIDCC=AKEyXzVORUAsxRgQaswJv9U474TK67FTluaTIT4T9FbZwMy_rVyl4jrSpiO_zYnR87k6xR96MoNmKA',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': 'ab865df8-5652-4545-8d4b-0d8c16d71bcf',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '4f53dde5-ccd3-47a8-a13a-1683958b3094',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_u5wcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_vfpcznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                            'transformationConfig': {
                                'transformationType': 0,
                            },
                        },
                    },
                    {
                        'name': 'qt_duucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_euucznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_duucznzjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_9p89ss7ihc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'GUADA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_mspdznzjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'abr 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_vs1cznzjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}


In [41]:
gdl_venta = run_looker_pipeline(
    cookies, headers, params, json_data,
    pais="Mexico",
    ciudad="Guadalajara",
    operacion="sell",
    columnas={
        "col_0": "neighborhood",
        "col_1": "new",
        "col_2": "index",
        "col_3": "used"
    },
     mes_field_name="_fecha_"
)
print("Total de registros", len(gdl_venta))
gdl_venta.to_csv("gdl_venta.csv", encoding="latin1",index=False)
gdl_venta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,FRACC PUERTA DE HIERRO,64752.0,61989,57814.0,ago. 21,Mexico,Guadalajara,sell
1,RES VIRREYES,60540.0,59004,46076.0,ago. 21,Mexico,Guadalajara,sell
2,COLINAS DE SAN JAVIER 2DA SECC,NaN,53318,NaN,ago. 21,Mexico,Guadalajara,sell
3,PROVIDENCIA 4TA SECC,51821.0,51199,NaN,ago. 21,Mexico,Guadalajara,sell
4,COUNTRY CLUB,48858.0,48324,47747.0,ago. 21,Mexico,Guadalajara,sell
...,...,...,...,...,...,...,...,...
985,SAN JUAN DE DIOS,35261.0,34029,33526.0,feb. 25,Mexico,Guadalajara,sell
986,FRACC LAS ARBOLEDAS,NaN,33743,33835.0,feb. 25,Mexico,Guadalajara,sell
987,SANTA ANA TEPETITLAN,31013.0,30260,NaN,feb. 25,Mexico,Guadalajara,sell
988,LOMAS DE INDEPENDENCIA,NaN,29910,NaN,feb. 25,Mexico,Guadalajara,sell


In [42]:
cookies = {
    'RAP_XSRF_TOKEN': 'AImk1AKLsvXWVZMCYumMBVBVU1yAdeKzlA:1774509296273',
    '__Secure-3PSID': 'g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076',
    '__Secure-3PAPISID': '4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2',
    'NID': '530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-',
    '__Secure-3PSIDTS': 'sidts-CjEBWhotCU1tqP2KIR7sREia1cfurcOv1e3OzKW4E_VGxQDPoJ8qzl5mzUPcGJBOyfDZEAA',
    '__Secure-3PSIDCC': 'AKEyXzX0p4uYlEBEDvK8tgbux9VkbvRWv6IIeHK-Sfv3XKEvmPYRIRNJwMa3h5OsNxCsgbbhN_2OJg',
}

headers = {
    'accept': 'application/json, text/plain, */*',
    'accept-language': 'es-US,es-419;q=0.9,es;q=0.8',
    'content-type': 'application/json',
    'encoding': 'null',
    'origin': 'https://lookerstudio.google.com',
    'priority': 'u=1, i',
    'referer': 'https://lookerstudio.google.com/embed/u/0/reporting/fecf012d-793d-4dc8-af3a-aa69f96b68c5/page/Ibx9',
    'sec-ch-ua': '"Chromium";v="146", "Not-A.Brand";v="24", "Google Chrome";v="146"',
    'sec-ch-ua-mobile': '?0',
    'sec-ch-ua-platform': '"Windows"',
    'sec-fetch-dest': 'empty',
    'sec-fetch-mode': 'cors',
    'sec-fetch-site': 'same-origin',
    'sec-fetch-storage-access': 'active',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36',
    'x-browser-channel': 'stable',
    'x-browser-copyright': 'Copyright 2026 Google LLC. All Rights reserved.',
    'x-browser-validation': 'NNdAQ4umZKwG7+IzfjmwHCgaZsw=',
    'x-browser-year': '2026',
    'x-client-data': 'CJC2yQEIo7bJAQipncoBCNmOywEIk6HLAQiFoM0BGLGKzwEY7LHPAQ==',
    'x-rap-xsrf-token': 'AImk1AKLsvXWVZMCYumMBVBVU1yAdeKzlA:1774509296273',
    # 'cookie': 'RAP_XSRF_TOKEN=AImk1AKLsvXWVZMCYumMBVBVU1yAdeKzlA:1774509296273; __Secure-3PSID=g.a0008AjW0fizYqF_OW2PjPUdCZsj-bTx19tWwK5ERllhmHv-or2zwzML6ufATVqqfykeGrJX2QACgYKAWISARASFQHGX2Mi07GCjxdAoFr3dmxGWjN_BBoVAUF8yKqQ8LQldxrukW8TAOyqVPQr0076; __Secure-3PAPISID=4dDtZvB4sigpDiLn/AOT7_UBLRZtm2V6k2; NID=530=KHDufgHFCQLknYnyJjdIEjj-HUUAd61ubTnZ-BWrYvh5u5V4t0QuzqEb-U494hDrvokee2JIJ7GHcB-Bud4Wp8vEgV_A4bFqzTCf1pHU1Z9TIaXj6n3axHw7KeSTo063AWOdkLtnj8xgfUcDZGbz3E4T2fbqcMy8EAwwDbFG0WHmypXREaXt0YQG7_QpGF9w5-kEynynmiF8K5LoJG6WloRmUTdcl2l8CLDobzvDKapvYazKMrBu6djhQO5LfU1uxaazmKGJyRZqRCAllhWbd8NFB1JkMNurSgXCf48X2eizK9Xwmq1PfE2RtoOWu2N1UKEqT8K-UU98jYefj23VGoA31Ab4plyCaGM1DfhpfRLaV8YSzl5dsA_TYtWfhpkr8b31uZhQzoHoT9gJvyNZM6GH5c_HhZTVs47p-uPVoX5VtBDHp2qDAbsu5nfGZDE7_6FyLytc30SxbuQI27ln7P2DGhQmrPKIkXGTYtdvsuYKTCmDhwQ-ahovfhvJE3PvDW2pU9UOlYHmUbNJl096vF3GG9IheCKqmvwMsuUuOukJwbKGUOU372tgH0fHFIWDlG9QhQu8jRcob_apIqNKqbcVLjwZgPKFrfblzzd6GSyOBg01bHG5HBd5EhGGYvNtwEzy7WkSp5Ze6AOq23MQFvsir-38qO9AkF_qljA_nao9hpkzmesZzRSqDGx07XcsHxRdmsd1ixucNFDiRqXuZ0vUjhhZ83673D9Yjq-UK86FPT-5daZbKkGBtTC__QTEwCp4dbTsycau5Di_w7vq4Hmky0SI2JH44BeHahxOs317VKPoXt7F4QuChSNwfhIlIeSJ6RSqj_byg1RJR8CYZlEtJySgL-fHdLpuDQhCfs-CzAc458NafLgYiM_LSIY2EUft-lNz8Q8lfAGlSL-8cUiiSzCc9cflG2xiKThJb-F_mhdRcnjVnALw7bYLRkCokVsC4AA2GxynRW0aY3w_jNQBXmpZ8VQaMd4Bn3T_Zx2GZ49mxWuzQXFrFyJhOJJW6aMLLj7Tob_oX_rXhaNjPLG6U3aI8ejyPL8KD3asAZwsrozI3bAmLlCbyOQvPQR6Fgo_QN4Jk9go6gyDIGMIPsJmELP_iayWedPPlYlD5h0dyyK_T1V8OfSFtNpQ81qDllMwCf9AhPgF2yB3D2bnmrG-Fzul3mq4AxUbOCtzq9jUpje-yACpO3gUcy-XWcRtCDrOZdgNfA81Yyau0kWdbahIG3pvoUXsh4XcUpMps5O9Flqw8Y0HGThsMF6cSL9hbcHL0iTQVbHpWUjD9H4jFHUufR0hhMvKuP2zyusVFqXK3nAUe3wJepJoC7vqH8JU6h8a6ZgnLvYBPSenTyvP1L7d77Ne0n99-ANUqXTYFg5hmLyx4g_-oL-xBu_aWcV0Hieb9d4uyDA48tnsKmPRTwBwLpOSRDr6wCL6SYZ8s44tPDMhc4rPRQHZk-3QVimdfPbOhV4gLHJNipONQMJOuAxpbrwER4QNs5L7VhjRHKl4k4SikUTa1TQV3hybpu4twMH1Yz0jktaJC85xV8LZOo5l4TiB77qhKnKhmffMbVcFptDoFUZWHLjr6tsEtkbZu68FhjfiMCXauyqQ1h245xwEKbGz-dVy8LVVUYWdqz2qbazz9O6b-xTTlVjH1zRES3ihM7pPSzolEh8eqmNJ_dfR6ITgN7QOISmHjdNVkmouJkjFdmg0OtRF66wuvotO6JGXImkg0jpZg7AMzitiXYwdL42Yz-tQeA8f3QYK2-_DDmpn6FyiIiA8Qc5cAOD3UFmMmvIPcH7UBD1ef_MWo81sYFhJ_Az2ymSY3sKbhD_79SK4padcxERHE2aqj2fenU30TTBHiPaIANe3P1ajuTydwSI8wWgDwPUgqkVcRRivKeDTlBPEZfwQMciSpIMAQu0xdKxAez7AFeub7Eb90iN0SpksHgVHlwhboxZaEphhVZUeFZsiE-3kXCx_DEltC24Pig1-jbQzb0kxa4CE-mT5H6cAabEpuDVQG_wxXCAqkk5aDDsyNLG-Cp0GDflPm7jymDCtMNamciwSJcoCByeA2ERsU66-r-kN4UNJrfKZd4PY9rrjajvsRPTgQ-7ohyj0-eNIChurHEhWOUf-0mSE7-EqdQzpEad-; __Secure-3PSIDTS=sidts-CjEBWhotCU1tqP2KIR7sREia1cfurcOv1e3OzKW4E_VGxQDPoJ8qzl5mzUPcGJBOyfDZEAA; __Secure-3PSIDCC=AKEyXzX0p4uYlEBEDvK8tgbux9VkbvRWv6IIeHK-Sfv3XKEvmPYRIRNJwMa3h5OsNxCsgbbhN_2OJg',
}

params = {
    'appVersion': '20260315_0000',
}

json_data = {
    'dataRequest': [
        {
            'requestContext': {
                'reportContext': {
                    'reportId': 'fecf012d-793d-4dc8-af3a-aa69f96b68c5',
                    'pageId': '14728046',
                    'mode': 1,
                    'componentId': 'cd-75b23orndc',
                    'displayType': 'simple-table',
                    'actionId': 'crossFilters',
                },
                'requestMode': 0,
            },
            'datasetSpec': {
                'dataset': [
                    {
                        'datasourceId': '8195b1fb-609b-4c7d-afc8-88810deb3e48',
                        'revisionNumber': 0,
                        'parameterOverrides': [],
                    },
                ],
                'queryFields': [
                    {
                        'name': 'qt_imb9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_nombre_n3_',
                        },
                    },
                    {
                        'name': 'qt_oz68k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_estrenar_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_1a98k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_index_precio_',
                            'aggregation': 1,
                        },
                    },
                    {
                        'name': 'qt_u298k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_usado_precio_',
                            'aggregation': 1,
                        },
                    },
                ],
                'sortData': [
                    {
                        'sortColumn': {
                            'name': 'qt_1a98k9zjic',
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'dataTransformation': {
                                'sourceFieldName': '_index_precio_',
                                'aggregation': 1,
                            },
                        },
                        'sortDir': 1,
                    },
                ],
                'includeRowsCount': True,
                'relatedDimensionMask': {
                    'addDisplay': False,
                    'addUniqueId': False,
                    'addLatLong': False,
                },
                'paginateInfo': {
                    'startRow': 1,
                    'rowsCount': 1000,
                },
                'dsFilterOverrides': [],
                'filters': [
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': False,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_ho99vjcxac',
                                },
                                'filterConditionType': 'NU',
                                'stringValues': [
                                    '',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_nombre_n3_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'ns': 't0',
                                    'name': 'qt_6dd35chjhc',
                                },
                                'filterConditionType': 'EQ',
                                'stringValues': [
                                    'GUADA',
                                ],
                                'numberValues': [],
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_ciudad_',
                                    },
                                },
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                    },
                    {
                        'filterDefinition': {
                            'filterExpression': {
                                'include': True,
                                'conceptType': 0,
                                'concept': {
                                    'name': 'qt_lk69k9zjic',
                                    'ns': 't0',
                                },
                                'queryTimeTransformation': {
                                    'dataTransformation': {
                                        'sourceFieldName': '_fecha_',
                                    },
                                },
                                'filterConditionType': 'IN',
                                'stringValues': [
                                    'abr 25',
                                ],
                            },
                        },
                        'dataSubsetNs': {
                            'datasetNs': 'd0',
                            'tableNs': 't0',
                            'contextNs': 'c0',
                        },
                        'version': 3,
                        'isCanvasFilter': True,
                    },
                ],
                'features': [],
                'dateRanges': [],
                'contextNsCount': 1,
                'dateRangeDimensions': [
                    {
                        'name': 'qt_b1g9k9zjic',
                        'datasetNs': 'd0',
                        'tableNs': 't0',
                        'dataTransformation': {
                            'sourceFieldName': '_fecha_as_date_',
                            'transformationConfig': {
                                'transformationType': 5,
                            },
                        },
                    },
                ],
                'calculatedField': [],
                'needGeocoding': False,
                'geoFieldMask': [],
                'multipleGeocodeFields': [],
                'timezone': 'America/Mexico_City',
            },
            'role': 'main',
            'retryHints': {
                'useClientControlledRetry': True,
                'isLastRetry': False,
                'retryCount': 0,
                'originalRequestId': 'cd-75b23orndc_0_0',
            },
        },
    ],
}

In [43]:
gdl_renta = run_looker_pipeline(
    cookies, headers, params, json_data,
    pais="Mexico",
    ciudad="Guadalajara",
    operacion="rent",
    columnas={
        "col_0": "neighborhood",
        "col_1": "new",
        "col_2": "index",
        "col_3": "used"
    },
     mes_field_name="_fecha_"
)
print("Total de registros", len(gdl_renta))
gdl_renta.to_csv("gdl_renta.csv", encoding="latin1",index=False)
gdl_renta

📍 Usando filtro de mes en índice: 2
⚠️ ene. 15 sin datos
⚠️ feb. 15 sin datos
⚠️ mar. 15 sin datos
⚠️ abr. 15 sin datos
⚠️ may. 15 sin datos
⚠️ jun. 15 sin datos
⚠️ jul. 15 sin datos
⚠️ ago. 15 sin datos
⚠️ sep. 15 sin datos
⚠️ oct. 15 sin datos
⚠️ nov. 15 sin datos
⚠️ dic. 15 sin datos
⚠️ ene. 16 sin datos
⚠️ feb. 16 sin datos
⚠️ mar. 16 sin datos
⚠️ abr. 16 sin datos
⚠️ may. 16 sin datos
⚠️ jun. 16 sin datos
⚠️ jul. 16 sin datos
⚠️ ago. 16 sin datos
⚠️ sep. 16 sin datos
⚠️ oct. 16 sin datos
⚠️ nov. 16 sin datos
⚠️ dic. 16 sin datos
⚠️ ene. 17 sin datos
⚠️ feb. 17 sin datos
⚠️ mar. 17 sin datos
⚠️ abr. 17 sin datos
⚠️ may. 17 sin datos
⚠️ jun. 17 sin datos
⚠️ jul. 17 sin datos
⚠️ ago. 17 sin datos
⚠️ sep. 17 sin datos
⚠️ oct. 17 sin datos
⚠️ nov. 17 sin datos
⚠️ dic. 17 sin datos
⚠️ ene. 18 sin datos
⚠️ feb. 18 sin datos
⚠️ mar. 18 sin datos
⚠️ abr. 18 sin datos
⚠️ may. 18 sin datos
⚠️ jun. 18 sin datos
⚠️ jul. 18 sin datos
⚠️ ago. 18 sin datos
⚠️ sep. 18 sin datos
⚠️ oct. 18 sin dato

,neighborhood,new,index,used,mes,pais,ciudad,operacion
0,RES VIRREYES,17520.0,15719,10932.0,ago. 21,Mexico,Guadalajara,rent
1,FRACC PUERTA DE HIERRO,16780.0,15639,14811.0,ago. 21,Mexico,Guadalajara,rent
2,HACIENDA DE LAS LOMAS,13278.0,13516,NaN,ago. 21,Mexico,Guadalajara,rent
3,FRACC JARDINES UNIVERSIDAD,14159.0,13430,10692.0,ago. 21,Mexico,Guadalajara,rent
4,FRACC LOMAS ALTAS,13608.0,13315,12983.0,ago. 21,Mexico,Guadalajara,rent
...,...,...,...,...,...,...,...,...
364,FRACC JARDINES DE LA PATRIA,NaN,13494,NaN,feb. 25,Mexico,Guadalajara,rent
365,FRACC ARCOS DE GUADALUPE,NaN,13464,13464.0,feb. 25,Mexico,Guadalajara,rent
366,ZAPOPAN,13165.0,12984,12900.0,feb. 25,Mexico,Guadalajara,rent
367,LA AURORA,NaN,11971,11606.0,feb. 25,Mexico,Guadalajara,rent
